# Fine-tuning de Meta omniASR-CTC-300M sur le baoulé

Ce notebook Kaggle présente le fine-tuning de `omniASR_CTC_300M` avec la recette CTC officielle de Meta et Fairseq2. L’entraînement utilise le tokenizer `omniASR_tokenizer_v1`, la loss CTC, AdamW et le WER pour suivre les résultats.

Les données viennent de `google/WaxalNLP`, configuration `bau_tts`. Seuls les audios de **2 à 60 secondes** sont conservés. Les enregistrements plus longs sont exclus, car leur découpage demanderait des timestamps alignés avec le texte.

Une première expérience de 500 pas sert de référence, suivie d’une nouvelle expérience de 1000 pas sur les mêmes données.

Sources : [recette CTC Meta](https://github.com/facebookresearch/omnilingual-asr/tree/main/workflows/recipes/wav2vec2/asr) et [préparation des données](https://github.com/facebookresearch/omnilingual-asr/tree/main/workflows/dataprep).


In [ ]:
!git clone --depth 1 https://github.com/facebookresearch/omnilingual-asr.git /kaggle/working/omnilingual-asr
%pip install -q -e "/kaggle/working/omnilingual-asr[data]"
%pip install -q --no-cache-dir --force-reinstall --no-deps torchaudio==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu128
print("Installation terminée. Redémarre maintenant la session Kaggle, puis saute cette cellule.")


Après l’installation, redémarrer la session Kaggle et reprendre à la cellule suivante. La cellule d’installation ne doit pas être relancée.


In [ ]:
from pathlib import Path
import gc, io, json, os, re, shutil, subprocess, sys, unicodedata
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["HF_HOME"] = "/kaggle/working/hf-cache"

REPO_DIR = Path("/kaggle/working/omnilingual-asr")
DATASET_ID = "google/WaxalNLP"
DATASET_CONFIG = "bau_tts"
DATASET_BASE = Path("/kaggle/working/waxal-bau-meta")
DATASET_VERSION = DATASET_BASE / "version=0"
STATS_PATH = DATASET_BASE / "language_distribution_0.tsv"
ASSET_DIR = Path("/kaggle/working/waxal-assets")
TRAIN_CONFIG_PATH = Path("/kaggle/working/waxal-bau-ctc-300m.yaml")
BASELINE_CONFIG_PATH = Path("/kaggle/working/waxal-bau-ctc-300m-baseline.yaml")
TRAIN_OUTPUT = Path("/kaggle/working/omniASR-CTC-300M-bau-official")
BASELINE_OUTPUT = Path("/kaggle/working/omniASR-CTC-300M-bau-baseline")
MIN_AUDIO_SECONDS = 2.0
MAX_AUDIO_SECONDS = 60.0
NUM_STEPS = 500
LEARNING_RATE = 1e-5
GRAD_ACCUMULATION = 4
SEED = 42
for directory in (DATASET_BASE, ASSET_DIR, TRAIN_OUTPUT, BASELINE_OUTPUT):
    directory.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.is_dir():
    raise FileNotFoundError("Le dépôt Meta manque. Relance l'installation puis redémarre la session.")


In [ ]:
import torch, torchaudio, torchvision
from importlib.metadata import version

if not torch.cuda.is_available():
    raise RuntimeError("Active un GPU dans les Settings Kaggle.")
if not torch.__version__.startswith("2.8.") or not torchaudio.__version__.startswith("2.8."):
    raise RuntimeError(f"Versions incompatibles : torch={torch.__version__}, torchaudio={torchaudio.__version__}")
print("GPU :", torch.cuda.get_device_name(0))
print("VRAM :", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), "Gio")
print("torch :", torch.__version__, "| CUDA :", torch.version.cuda)
print("torchaudio :", torchaudio.__version__, "| torchvision :", torchvision.__version__)
print("omnilingual-asr :", version("omnilingual-asr"), "| fairseq2 :", version("fairseq2"))


## Préparer WaxalNLP au format Meta

Le sous-ensemble `bau_tts` est chargé puis préparé pour le CTC. Les annotations entre crochets qui ne sont pas prononcées sont retirées. Le texte passe en minuscules tout en conservant les lettres, les accents et les apostrophes du baoulé.

Les audios sont ensuite convertis en FLAC mono 16 kHz et enregistrés au format officiel `MixtureParquet`.


In [ ]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import soundfile as sf
from datasets import Audio, DatasetDict, load_dataset
from fairseq2.data.tokenizers.hub import load_tokenizer

raw = load_dataset(DATASET_ID, DATASET_CONFIG)
raw = DatasetDict({name: ds.cast_column("audio", Audio(sampling_rate=16_000)) for name, ds in raw.items()})
print(raw)
tokenizer = load_tokenizer("omniASR_tokenizer_v1")
token_encoder = tokenizer.create_encoder()
token_decoder = tokenizer.create_decoder(skip_special_tokens=True)
UNK_IDX = getattr(tokenizer.vocab_info, "unk_idx", None)
print("Token inconnu :", UNK_IDX)


In [ ]:
def normalize_ctc_text(text):
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"\[[^\]]*\]", " ", text)  # valeurs explicatives non prononcées
    text = text.replace("’", "'").replace("‘", "'").lower()
    chars = []
    for char in text:
        category = unicodedata.category(char)
        chars.append(char if char == "'" or char.isspace() or category[0] in {"L", "M", "N"} else " ")
    text = " ".join("".join(chars).split())
    text = re.sub(r"(?<!\S)\d+(?!\S)", " ", text)  # recommandation Meta : retirer les mots uniquement numériques
    return " ".join(text.split())

schema = pa.schema([
    pa.field("text", pa.string()),
    pa.field("audio_bytes", pa.list_(pa.int8())),
    pa.field("audio_size", pa.int64()),
])
split_mapping = {"validation": "dev"}
report = {}
all_hours = 0.0

def flush_rows(rows, output_file):
    table = pa.Table.from_pylist(rows, schema=schema)
    pq.write_table(table, output_file, compression="zstd", row_group_size=100)

marker = DATASET_BASE / "PREPARATION_COMPLETE.json"
if marker.is_file():
    report = json.loads(marker.read_text(encoding="utf-8"))
    print("Données déjà préparées :", report)
else:
    for source_split, ds in raw.items():
        target_split = split_mapping.get(source_split, source_split)
        output_dir = DATASET_VERSION / "corpus=waxal" / f"split={target_split}" / "language=bci_Latn"
        output_dir.mkdir(parents=True, exist_ok=True)
        rows, file_index, kept, rejected_duration, rejected_text = [], 0, 0, 0, 0
        seconds_kept = 0.0
        for index, item in enumerate(ds):
            audio = item["audio"]
            waveform = np.asarray(audio["array"], dtype=np.float32)
            if waveform.ndim == 2:
                waveform = waveform.mean(axis=0 if waveform.shape[0] < waveform.shape[1] else 1)
            waveform = np.ascontiguousarray(waveform.reshape(-1), dtype=np.float32)
            duration = len(waveform) / 16_000
            if not MIN_AUDIO_SECONDS <= duration <= MAX_AUDIO_SECONDS:
                rejected_duration += 1
                continue
            text = normalize_ctc_text(item["text"])
            if not text:
                rejected_text += 1
                continue
            encoded = token_encoder(text)
            decoded = token_decoder(encoded)
            if (UNK_IDX is not None and UNK_IDX in encoded.tolist()) or "⁇" in decoded:
                rejected_text += 1
                continue
            buffer = io.BytesIO()
            sf.write(buffer, waveform, 16_000, format="FLAC")
            audio_int8 = np.frombuffer(buffer.getvalue(), dtype=np.int8).tolist()
            rows.append({"text": text, "audio_bytes": audio_int8, "audio_size": len(waveform)})
            kept += 1; seconds_kept += duration
            if len(rows) >= 25:
                flush_rows(rows, output_dir / f"part-{file_index:05d}.parquet")
                rows = []; file_index += 1
            if (index + 1) % 100 == 0:
                print(f"{source_split}: {index + 1}/{len(ds)} lus, {kept} conservés")
        if rows:
            flush_rows(rows, output_dir / f"part-{file_index:05d}.parquet")
        report[target_split] = {"source_examples": len(ds), "kept": kept, "rejected_duration": rejected_duration, "rejected_text": rejected_text, "hours": seconds_kept / 3600}
        all_hours += seconds_kept / 3600
    pd.DataFrame([{"corpus": "waxal", "language": "bci_Latn", "hours": all_hours}]).to_csv(STATS_PATH, sep="\t", index=False)
    marker.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(report, ensure_ascii=False, indent=2))

if report.get("train", {}).get("kept", 0) == 0 or report.get("dev", {}).get("kept", 0) == 0:
    raise RuntimeError("Les splits train et dev doivent contenir des exemples valides.")
print(STATS_PATH.read_text(encoding="utf-8"))

# Le Parquet Meta est désormais autonome : supprimer la copie source du dataset.
# Le cache Hub qui contient le modèle et le tokenizer est conservé.
if "raw" in globals():
    del raw
gc.collect()
shutil.rmtree(Path(os.environ["HF_HOME"]) / "datasets", ignore_errors=True)
print("Cache source du dataset supprimé après conversion.")


In [ ]:
dataset_card = f'''name: waxal_bau
dataset_family: mixture_parquet_asr_dataset
dataset_config:
  data: {DATASET_VERSION}
tokenizer_ref: omniASR_tokenizer_v1
'''
(ASSET_DIR / "waxal_bau.yaml").write_text(dataset_card, encoding="utf-8")
max_audio_samples = int(MAX_AUDIO_SECONDS * 16_000)
min_audio_samples = int(MIN_AUDIO_SECONDS * 16_000)

train_yaml = f'''model:
  name: "omniASR_CTC_300M"
dataset:
  name: "waxal_bau"
  train_split: "train"
  valid_split: "dev"
  storage_mode: "MIXTURE_PARQUET"
  task_mode: "ASR"
  mixture_parquet_storage_config:
    dataset_summary_path: "{STATS_PATH}"
    beta_corpus: 0.5
    beta_language: 0.5
    fragment_loading:
      cache: false
  asr_task_config:
    min_audio_len: {min_audio_samples}
    max_audio_len: {max_audio_samples}
    max_num_elements: {max_audio_samples}
    batch_shuffle_window: 1
    normalize_audio: true
    example_shuffle_window: 1000
tokenizer:
  name: "omniASR_tokenizer_v1"
optimizer:
  config:
    lr: {LEARNING_RATE}
trainer:
  freeze_encoder_for_n_steps: 0
  mixed_precision:
    dtype: "torch.float16"
  grad_accumulation:
    num_batches: {GRAD_ACCUMULATION}
  activation_checkpointing:
    mode: "layerwise"
    every_nth_layer: 1
  max_grad_norm: 1.0
regime:
  num_steps: {NUM_STEPS}
  score_metric: "wer"
  validate_after_n_steps: 100
  validate_every_n_steps: 100
  checkpoint_after_n_steps: 100
  checkpoint_every_n_steps: 100
  publish_metrics_every_n_steps: 20
  save_model_only: "all"
  keep_last_n_checkpoints: 1
  keep_best_n_checkpoints: 1
common:
  seed: {SEED}
  assets:
    extra_paths:
      - "{ASSET_DIR}"
'''
TRAIN_CONFIG_PATH.write_text(train_yaml, encoding="utf-8")

baseline_yaml = f'''model:
  name: "omniASR_CTC_300M"
tokenizer:
  name: "omniASR_tokenizer_v1"
dataset:
  name: "waxal_bau"
  valid_split: "dev"
  storage_mode: "MIXTURE_PARQUET"
  task_mode: "ASR"
  mixture_parquet_storage_config:
    dataset_summary_path: "{STATS_PATH}"
    beta_corpus: 0.5
    beta_language: 0.5
    fragment_loading:
      cache: false
  asr_task_config:
    min_audio_len: {min_audio_samples}
    max_audio_len: {max_audio_samples}
    max_num_elements: {max_audio_samples}
    batch_shuffle_window: 1
    normalize_audio: true
    example_shuffle_window: 1
evaluator:
  amp: true
  amp_dtype: "torch.float16"
common:
  seed: {SEED}
  assets:
    extra_paths:
      - "{ASSET_DIR}"
'''
BASELINE_CONFIG_PATH.write_text(baseline_yaml, encoding="utf-8")
print(TRAIN_CONFIG_PATH.read_text(encoding="utf-8"))


## Vérifier le chargement des données

Le dataloader officiel de Meta lit deux batches pour vérifier que les données sont valides. Toute erreur doit être corrigée avant de lancer l’entraînement.


In [ ]:
command = [
    sys.executable, "-m", "workflows.dataprep.dataloader_example",
    f"--dataset_path={DATASET_VERSION}", "--split=train", "--num_iterations=2",
]
print("Commande :", " ".join(map(str, command)))
subprocess.run(command, cwd=REPO_DIR, check=True)


## Mesurer le modèle de base sur `dev`

Les performances du modèle 300M original sont mesurées sur `dev` avant le fine-tuning. Cette baseline servira à évaluer les progrès des modèles entraînés.


In [ ]:
baseline_command = [
    sys.executable, "-m", "workflows.recipes.wav2vec2.asr.eval",
    str(BASELINE_OUTPUT), "--config-file", str(BASELINE_CONFIG_PATH),
]
print("Commande :", " ".join(baseline_command))
subprocess.run(baseline_command, cwd=REPO_DIR, check=True)


## Lancer la première expérience CTC — 500 pas

Tous les paramètres du modèle 300M sont entraînés, avec une évaluation sur `dev` tous les 100 pas. Pour respecter la limite de stockage de Kaggle, les checkpoints contiennent uniquement les poids du modèle. Une reprise exacte de l’optimiseur après une interruption n’est donc pas possible.


In [ ]:
active_train_yaml = TRAIN_CONFIG_PATH.read_text(encoding="utf-8")
if "example_shuffle_window: 1000" not in active_train_yaml:
    raise RuntimeError("YAML périmé : régénère-le avec example_shuffle_window: 1000.")
if 'save_model_only: "all"' not in active_train_yaml:
    raise RuntimeError('YAML périmé : régénère-le avec save_model_only: "all".')
free_gib = shutil.disk_usage("/kaggle/working").free / 2**30
print(f"Espace disque libre avant entraînement : {free_gib:.2f} Gio")
if free_gib < 4.0:
    raise RuntimeError("Moins de 4 Gio libres : nettoie /kaggle/working avant l'entraînement.")

train_command = [
    sys.executable, "-m", "workflows.recipes.wav2vec2.asr",
    str(TRAIN_OUTPUT), "--config-file", str(TRAIN_CONFIG_PATH),
]
print("Commande :", " ".join(train_command))
subprocess.run(train_command, cwd=REPO_DIR, check=True)


## Examiner les fichiers produits

Cette cellule affiche les checkpoints, les métriques et les transcriptions générés par Fairseq2. Le split `dev` sert à sélectionner le meilleur checkpoint, tandis que le split `test` reste réservé à l’évaluation finale.


In [ ]:
files = sorted(path for path in TRAIN_OUTPUT.rglob("*") if path.is_file())
print(f"{len(files)} fichiers produits dans {TRAIN_OUTPUT}")
for path in files:
    print(f"{path.relative_to(TRAIN_OUTPUT)}  {path.stat().st_size / 2**20:.1f} Mio")
print("\nLes checkpoints ne contiennent que le modèle afin de respecter la limite de stockage Kaggle.")


## Limites pratiques

- En cas de manque de mémoire sur la T4, réduire `MAX_AUDIO_SECONDS` à 45, régénérer les YAML et relancer dans un nouveau dossier.
- Ne pas découper les audios de plus de 60 secondes sans timestamps alignés avec leur transcription.
- Comparer le WER, le CER et plusieurs transcriptions avant de publier un modèle.


## Tester la version 500 pas

Le checkpoint du pas 500 est testé sur dix audios jamais vus du split `test`, puis sur un audio personnel. Comme le pipeline Meta accepte au maximum 40 secondes par appel, les audios longs sont découpés en segments de 35 secondes.


In [ ]:
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
import torch

checkpoint_candidates = sorted(
    TRAIN_OUTPUT.glob("ws_*/checkpoints/step_500/model/pp_00/tp_00/sdp_00.pt"),
    key=lambda path: path.stat().st_mtime,
)
if not checkpoint_candidates:
    raise FileNotFoundError("Checkpoint step_500 introuvable.")
checkpoint_path = checkpoint_candidates[-1]
print("Checkpoint :", checkpoint_path)

finetuned_pipeline = ASRInferencePipeline(
    model_card="omniASR_CTC_300M",
    device="cuda",
    dtype=torch.float16,
)
state_dict = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
finetuned_pipeline.model.load_state_dict(state_dict, strict=True)
del state_dict
finetuned_pipeline.model.eval()
torch.cuda.empty_cache()
print("Modèle fine-tuné chargé correctement.")


In [ ]:
import editdistance
import pyarrow.parquet as pq

def corpus_error_rate(references, predictions, level="word"):
    total_edits = 0
    total_reference_units = 0
    for reference, prediction in zip(references, predictions):
        if level == "word":
            reference_units = reference.split()
            prediction_units = prediction.split()
        else:
            reference_units = list(reference)
            prediction_units = list(prediction)
        total_edits += editdistance.eval(reference_units, prediction_units)
        total_reference_units += len(reference_units)
    return total_edits / max(total_reference_units, 1)

N_TEST = 10
records = []
test_files = sorted((DATASET_VERSION / "corpus=waxal" / "split=test" / "language=bci_Latn").glob("*.parquet"))
for parquet_file in test_files:
    table = pq.read_table(parquet_file, columns=["text", "audio_bytes", "audio_size"])
    for row in table.to_pylist():
        # Le pipeline Meta refuse un segment individuel de plus de 40 secondes.
        if row["audio_size"] <= 40 * 16_000:
            records.append(row)
        if len(records) >= N_TEST:
            break
    if len(records) >= N_TEST:
        break

if not records:
    raise RuntimeError("Aucun exemple test de 40 secondes ou moins.")

audio_inputs = [bytes((value % 256 for value in row["audio_bytes"])) for row in records]
references = [row["text"] for row in records]
predictions = finetuned_pipeline.transcribe(audio_inputs, batch_size=1)

for index, (reference, prediction) in enumerate(zip(references, predictions), 1):
    print(f"\n--- Exemple {index} ---")
    print("RÉFÉRENCE :", reference)
    print("PRÉDICTION :", prediction)

print(f"\nWER sur ces {len(records)} audios : {100 * corpus_error_rate(references, predictions, 'word'):.2f} %")
print(f"CER sur ces {len(records)} audios : {100 * corpus_error_rate(references, predictions, 'character'):.2f} %")


Pour tester un audio personnel, remplacer uniquement `AUDIO_PATH`. Un fichier WAV ou FLAC est préférable. Les segments sont transcrits dans l’ordre puis réunis.


In [ ]:
import numpy as np
import soundfile as sf

AUDIO_PATH = "/kaggle/input/nom-du-dataset/mon-audio.wav"  # À MODIFIER
SEGMENT_SECONDS = 35

waveform, sample_rate = sf.read(AUDIO_PATH, dtype="float32", always_2d=True)
waveform = waveform.mean(axis=1)  # mono
segment_samples = SEGMENT_SECONDS * sample_rate
segments = [
    {"waveform": np.ascontiguousarray(waveform[start:start + segment_samples]), "sample_rate": sample_rate}
    for start in range(0, len(waveform), segment_samples)
    if len(waveform[start:start + segment_samples]) > 0
]

segment_predictions = finetuned_pipeline.transcribe(segments, batch_size=1)
for index, prediction in enumerate(segment_predictions, 1):
    print(f"Segment {index}: {prediction}")
print("\nTRANSCRIPTION COMPLÈTE :")
print(" ".join(segment_predictions))


## Publier la version 500 pas sur Hugging Face

Le dépôt est privé par défaut. Le checkpoint, sa configuration, ses métriques et sa fiche de modèle y sont envoyés directement. Utiliser `PRIVATE_REPO = False` pour le rendre public.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from huggingface_hub import HfApi
from pathlib import Path
import io

REPO_ID = "DEFINI TON REPO_ID"
PRIVATE_REPO = True
api = HfApi()
api.create_repo(REPO_ID, repo_type="model", private=PRIVATE_REPO, exist_ok=True)

checkpoint_candidates = sorted(
    TRAIN_OUTPUT.glob("ws_*/checkpoints/step_500/model/pp_00/tp_00/sdp_00.pt"),
    key=lambda path: path.stat().st_mtime,
)
if not checkpoint_candidates:
    raise FileNotFoundError("Checkpoint step_500 introuvable.")
checkpoint_path = checkpoint_candidates[-1]
run_dir = checkpoint_path.parents[5]

# vous pouvez modifier le readme!
readme_lines = [
    "---",
    "library_name: fairseq2",
    "license: apache-2.0",
    "base_model: facebook/omniASR-CTC-300M",
    "datasets:",
    "- google/WaxalNLP",
    "language:",
    "- bci",
    "tags:",
    "- automatic-speech-recognition",
    "- ctc",
    "- baoule",
    "---",
    "",
    "# omniASR-CTC-300M Baoulé",
    "",
    "Fine-tuning complet de `facebook/omniASR-CTC-300M` avec la recette CTC officielle Meta sur `google/WaxalNLP` (`bau_tts`).",
    "",
    "- Étapes : 500",
    "- WER dev : 36.7576 %",
    "- UER dev : 11.8879 %",
    "- Tokenizer : `omniASR_tokenizer_v1` inchangé",
    "- Audio d'entraînement : 2 à 60 secondes",
    "",
    "Le fichier `model.pt` est un state dict Fairseq2. Il se charge dans l'architecture `omniASR_CTC_300M`.",
]
readme = "\n".join(readme_lines) + "\n"

api.upload_file(
    path_or_fileobj=io.BytesIO(readme.encode("utf-8")),
    path_in_repo="README.md", repo_id=REPO_ID, repo_type="model",
    commit_message="DEFINI TON MESSAGE DE COMMIT",
)
api.upload_file(
    path_or_fileobj=str(checkpoint_path),
    path_in_repo="model.pt", repo_id=REPO_ID, repo_type="model",
    commit_message="Upload 500-step fine-tuned checkpoint",
)

for local_path, hub_path in [
    (run_dir / "config.yaml", "training/config.yaml"),
    (run_dir / "checkpoints/model.yaml", "training/model.yaml"),
    (run_dir / "metrics/train.jsonl", "metrics/train.jsonl"),
    (run_dir / "metrics/valid.jsonl", "metrics/valid.jsonl"),
    (run_dir / "transcriptions/rank_0.ref.txt", "evaluation/dev.ref.txt"),
    (run_dir / "transcriptions/rank_0.hyp.txt", "evaluation/dev.hyp.txt"),
]:
    if local_path.is_file():
        api.upload_file(
            path_or_fileobj=str(local_path), path_in_repo=hub_path,
            repo_id=REPO_ID, repo_type="model", commit_message=f"Add {hub_path}",
        )

info = api.model_info(REPO_ID)
print("Modèle envoyé :", f"https://huggingface.co/{info.id}")


## Lancer une nouvelle expérience de 1000 pas

Le checkpoint 500 contient uniquement les poids, sans l’état de l’optimiseur ni du scheduler. Pour garder un entraînement cohérent sur 1000 pas, cette expérience repart du modèle Meta original. Fairseq2 conserve le meilleur checkpoint selon le WER sur `dev`.


In [ ]:
import gc
import re
import shutil
from pathlib import Path

if 'info' not in globals() or info.id != REPO_ID:
    raise RuntimeError("L'envoi Hugging Face doit réussir avant le nettoyage.")
globals().pop("finetuned_pipeline", None)
gc.collect()
torch.cuda.empty_cache()

# Nettoyage après le succès confirmé de l'envoi Hugging Face.
OLD_TRAIN_OUTPUT = TRAIN_OUTPUT
shutil.rmtree(OLD_TRAIN_OUTPUT, ignore_errors=True)

NUM_STEPS = 1000
TRAIN_OUTPUT = Path("/kaggle/working/omniASR-CTC-300M-bau-official-1000")
TRAIN_CONFIG_PATH = Path("/kaggle/working/waxal-bau-ctc-300m-1000.yaml")
TRAIN_OUTPUT.mkdir(parents=True, exist_ok=True)

source_config_path = Path("/kaggle/working/waxal-bau-ctc-300m.yaml")
config_1000 = source_config_path.read_text(encoding="utf-8")
config_1000 = re.sub(r"num_steps:\s*500\b", "num_steps: 1000", config_1000, count=1)
if "num_steps: 1000" not in config_1000:
    raise RuntimeError("Impossible de définir num_steps à 1000.")
if "example_shuffle_window: 1000" not in config_1000:
    raise RuntimeError("example_shuffle_window incorrect.")
if 'save_model_only: "all"' not in config_1000:
    raise RuntimeError("save_model_only incorrect.")
TRAIN_CONFIG_PATH.write_text(config_1000, encoding="utf-8")

free_gib = shutil.disk_usage("/kaggle/working").free / 2**30
print(f"Espace libre avant les 1000 pas : {free_gib:.2f} Gio")
if free_gib < 4.0:
    raise RuntimeError("Il faut au moins 4 Gio libres avant de relancer.")
print(TRAIN_CONFIG_PATH.read_text(encoding="utf-8"))


In [ ]:
train_command_1000 = [
    sys.executable, "-m", "workflows.recipes.wav2vec2.asr",
    str(TRAIN_OUTPUT), "--config-file", str(TRAIN_CONFIG_PATH),
]
print("Commande :", " ".join(train_command_1000))
subprocess.run(train_command_1000, cwd=REPO_DIR, check=True)


## Tester la version 1000 pas

Le checkpoint du pas 1000 obtient le meilleur WER sur `dev`. Il est testé sur les mêmes dix audios que la version 500 afin de comparer les deux modèles dans les mêmes conditions.


In [ ]:
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
import gc
import torch

# Le journal confirme que step_1000 est le meilleur checkpoint.
checkpoint_1000_candidates = sorted(
    TRAIN_OUTPUT.glob("ws_*/checkpoints/step_1000/model/pp_00/tp_00/sdp_00.pt"),
    key=lambda path: path.stat().st_mtime,
)
if not checkpoint_1000_candidates:
    raise FileNotFoundError("Checkpoint step_1000 introuvable.")
checkpoint_1000 = checkpoint_1000_candidates[-1]
print("Checkpoint :", checkpoint_1000)

globals().pop("finetuned_pipeline", None)
globals().pop("finetuned_pipeline_1000", None)
gc.collect()
torch.cuda.empty_cache()

finetuned_pipeline_1000 = ASRInferencePipeline(
    model_card="omniASR_CTC_300M",
    device="cuda",
    dtype=torch.float16,
)
state_dict_1000 = torch.load(checkpoint_1000, map_location="cpu", weights_only=True)
finetuned_pipeline_1000.model.load_state_dict(state_dict_1000, strict=True)
del state_dict_1000
finetuned_pipeline_1000.model.eval()
torch.cuda.empty_cache()
print("Version 1000 pas chargée correctement.")


In [ ]:
import editdistance
import pyarrow.parquet as pq

def corpus_error_rate(references, predictions, level="word"):
    total_edits = 0
    total_reference_units = 0
    for reference, prediction in zip(references, predictions):
        if level == "word":
            reference_units = reference.split()
            prediction_units = prediction.split()
        else:
            reference_units = list(reference)
            prediction_units = list(prediction)
        total_edits += editdistance.eval(reference_units, prediction_units)
        total_reference_units += len(reference_units)
    return total_edits / max(total_reference_units, 1)

N_TEST = 10
records_1000 = []
test_dir = DATASET_VERSION / "corpus=waxal" / "split=test" / "language=bci_Latn"
for parquet_file in sorted(test_dir.glob("*.parquet")):
    table = pq.read_table(parquet_file, columns=["text", "audio_bytes", "audio_size"])
    for row in table.to_pylist():
        if row["audio_size"] <= 40 * 16_000:
            records_1000.append(row)
        if len(records_1000) >= N_TEST:
            break
    if len(records_1000) >= N_TEST:
        break

if not records_1000:
    raise RuntimeError("Aucun exemple test compatible trouvé.")

audio_inputs_1000 = [
    bytes(value % 256 for value in row["audio_bytes"])
    for row in records_1000
]
references_1000 = [row["text"] for row in records_1000]
predictions_1000 = finetuned_pipeline_1000.transcribe(audio_inputs_1000, batch_size=1)

for index, (reference, prediction) in enumerate(zip(references_1000, predictions_1000), 1):
    print(f"\n--- Exemple {index} ---")
    print("RÉFÉRENCE :", reference)
    print("PRÉDICTION 1000 :", prediction)

wer_1000 = 100 * corpus_error_rate(references_1000, predictions_1000, "word")
cer_1000 = 100 * corpus_error_rate(references_1000, predictions_1000, "character")
print(f"\nWER version 1000 sur ces {len(records_1000)} audios : {wer_1000:.2f} %")
print(f"CER version 1000 sur ces {len(records_1000)} audios : {cer_1000:.2f} %")
print(f"Comparaison version 500 : WER 72.22 % | CER 48.54 %")
print(f"Variation : WER {wer_1000 - 72.22:+.2f} points | CER {cer_1000 - 48.54:+.2f} points")


### Tester un audio personnel avec la version 1000

Remplacer uniquement `AUDIO_PATH`. Les audios longs sont automatiquement découpés en segments de 35 secondes.


In [ ]:
import numpy as np
import soundfile as sf

AUDIO_PATH = "/kaggle/input/nom-du-dataset/mon-audio.wav"  # À MODIFIER
SEGMENT_SECONDS = 35
waveform, sample_rate = sf.read(AUDIO_PATH, dtype="float32", always_2d=True)
waveform = waveform.mean(axis=1)
segment_samples = SEGMENT_SECONDS * sample_rate
segments = [
    {
        "waveform": np.ascontiguousarray(waveform[start:start + segment_samples]),
        "sample_rate": sample_rate,
    }
    for start in range(0, len(waveform), segment_samples)
]
predictions = finetuned_pipeline_1000.transcribe(segments, batch_size=1)
for index, prediction in enumerate(predictions, 1):
    print(f"Segment {index} : {prediction}")
print("\nTRANSCRIPTION COMPLÈTE :")
print(" ".join(predictions))


## Publier la version 1000 pas séparément

Le dépôt de la version 500 reste intact. La version 1000 est envoyée dans un second dépôt : `Tree-AI-lab/omniASR-CTC-300M-bau-1k`.


In [ ]:
from huggingface_hub import HfApi
from pathlib import Path
import io

REPO_1000 = "DEFINI TON REPO_ID"
PRIVATE_REPO = True
api = HfApi()
api.create_repo(REPO_1000, repo_type="model", private=PRIVATE_REPO, exist_ok=True)

output_1000 = Path("/kaggle/working/omniASR-CTC-300M-bau-official-1000")
candidates = sorted(
    output_1000.glob("ws_*/checkpoints/step_1000/model/pp_00/tp_00/sdp_00.pt"),
    key=lambda path: path.stat().st_mtime,
)
if not candidates:
    raise FileNotFoundError("Checkpoint step_1000 introuvable.")
checkpoint_1000 = candidates[-1]
run_dir_1000 = checkpoint_1000.parents[5]
assert "official-1000" in str(checkpoint_1000) and "step_1000" in str(checkpoint_1000)
print("Checkpoint :", checkpoint_1000)

readme_lines = [
    "---", "library_name: fairseq2", "license: apache-2.0",
    "base_model: facebook/omniASR-CTC-300M", "datasets:", "- google/WaxalNLP",
    "language:", "- bci", "tags:", "- automatic-speech-recognition", "- ctc", "- baoule", "---", "",
    "# omniASR-CTC-300M Baoulé — 1000 pas", "",
    "Fine-tuning complet avec la recette CTC officielle Meta sur `google/WaxalNLP` (`bau_tts`).", "",
    "- Étapes : 1000", "- Meilleur checkpoint : étape 1000",
    "- WER dev : 35.1525 %", "- UER dev : 10.9452 %", "- Exemples dev : 98",
    "- Tokenizer : `omniASR_tokenizer_v1` inchangé", "- Audio d'entraînement : 2 à 60 secondes", "",
    "Le fichier `model.pt` contient le state dict Fairseq2 du checkpoint 1000.",
]
readme = "\n".join(readme_lines) + "\n"

api.upload_file(
    path_or_fileobj=io.BytesIO(readme.encode("utf-8")), path_in_repo="README.md",
    repo_id=REPO_1000, repo_type="model", commit_message="Add 1000-step model card",
)
api.upload_file(
    path_or_fileobj=str(checkpoint_1000), path_in_repo="model.pt",
    repo_id=REPO_1000, repo_type="model", commit_message="DEFINI TON MESSAGE DE COMMIT",
)

for local_path, hub_path in [
    (run_dir_1000 / "config.yaml", "training/config.yaml"),
    (run_dir_1000 / "checkpoints/model.yaml", "training/model.yaml"),
    (run_dir_1000 / "checkpoints/scores/step_1000.txt", "evaluation/step_1000_score.txt"),
    (run_dir_1000 / "metrics/train.jsonl", "metrics/train.jsonl"),
    (run_dir_1000 / "metrics/valid.jsonl", "metrics/valid.jsonl"),
    (run_dir_1000 / "transcriptions/rank_0.ref.txt", "evaluation/dev.ref.txt"),
    (run_dir_1000 / "transcriptions/rank_0.hyp.txt", "evaluation/dev.hyp.txt"),
]:
    if local_path.is_file():
        api.upload_file(
            path_or_fileobj=str(local_path), path_in_repo=hub_path,
            repo_id=REPO_1000, repo_type="model", commit_message=f"Add {hub_path}",
        )

info_1000 = api.model_info(REPO_1000)
print("Version 1000 envoyée :", f"https://huggingface.co/{info_1000.id}")
print("Le dépôt 500 n'a pas été modifié.")

# conversion pour hf  transformers

In [ ]:
"""
Convertit le checkpoint fairseq2 fine-tuné à 1000 pas (omniASR_CTC_300M sur le
baoulé, meilleur WER : 35.15 %) en un modèle `Wav2Vec2ForCTC` natif transformers,
chargeable en 3 lignes :

    from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
    processor = Wav2Vec2Processor.from_pretrained("Tree-AI-lab/omniASR-CTC-300M-baoule-1000steps")
    model     = Wav2Vec2ForCTC.from_pretrained("Tree-AI-lab/omniASR-CTC-300M-baoule-1000steps")

À COLLER dans une nouvelle cellule de votre notebook Kaggle, APRÈS la cellule 30
(celle qui charge `finetuned_pipeline_1000` et définit `checkpoint_1000`). Réutilise
directement vos variables `checkpoint_1000`, `tokenizer`, `TRAIN_OUTPUT`, `REPO_1000`.

Logique de conversion adaptée de https://github.com/ahmedadelattia/omnilingual_to_hf
(licence : outil de conversion fourni "tel quel", poids omniASR sous Apache-2.0).
"""

import json
import logging
from pathlib import Path
from typing import Dict, List

import torch
from torch import Tensor

logging.basicConfig(format="%(levelname)s: %(message)s", level=logging.INFO)
logger = logging.getLogger("omni_to_hf")

# ─────────────────────────────────────────────────────────────────────────
# 1. Mapping des clés fairseq2 -> HuggingFace Wav2Vec2ForCTC
# ─────────────────────────────────────────────────────────────────────────
_STATIC_MAP: Dict[str, str] = {
    "encoder_frontend.post_extract_layer_norm.weight": "wav2vec2.feature_projection.layer_norm.weight",
    "encoder_frontend.post_extract_layer_norm.bias": "wav2vec2.feature_projection.layer_norm.bias",
    "encoder_frontend.model_dim_proj.weight": "wav2vec2.feature_projection.projection.weight",
    "encoder_frontend.model_dim_proj.bias": "wav2vec2.feature_projection.projection.bias",
    "encoder_frontend.pos_encoder.conv.bias": "wav2vec2.encoder.pos_conv_embed.conv.bias",
    "encoder_frontend.pos_encoder.conv.weight_g": "wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0",
    "encoder_frontend.pos_encoder.conv.weight_v": "wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1",
    "encoder.layer_norm.weight": "wav2vec2.encoder.layer_norm.weight",
    "encoder.layer_norm.bias": "wav2vec2.encoder.layer_norm.bias",
    "final_proj.weight": "lm_head.weight",
    "final_proj.bias": "lm_head.bias",
}

_LAYER_SUBKEY_MAP: Dict[str, str] = {
    "self_attn.q_proj.weight": "attention.q_proj.weight",
    "self_attn.q_proj.bias": "attention.q_proj.bias",
    "self_attn.k_proj.weight": "attention.k_proj.weight",
    "self_attn.k_proj.bias": "attention.k_proj.bias",
    "self_attn.v_proj.weight": "attention.v_proj.weight",
    "self_attn.v_proj.bias": "attention.v_proj.bias",
    "self_attn.output_proj.weight": "attention.out_proj.weight",
    "self_attn.output_proj.bias": "attention.out_proj.bias",
    "self_attn_layer_norm.weight": "layer_norm.weight",
    "self_attn_layer_norm.bias": "layer_norm.bias",
    "ffn.inner_proj.weight": "feed_forward.intermediate_dense.weight",
    "ffn.inner_proj.bias": "feed_forward.intermediate_dense.bias",
    "ffn.output_proj.weight": "feed_forward.output_dense.weight",
    "ffn.output_proj.bias": "feed_forward.output_dense.bias",
    "ffn_layer_norm.weight": "final_layer_norm.weight",
    "ffn_layer_norm.bias": "final_layer_norm.bias",
}


def build_key_mapping(n_layers: int) -> Dict[str, str]:
    mapping = dict(_STATIC_MAP)
    for i in range(7):  # 7 couches conv du feature extractor, fixe pour tous les OmniASR
        fs2_prefix = f"encoder_frontend.feature_extractor.layers.{i}"
        hf_prefix = f"wav2vec2.feature_extractor.conv_layers.{i}"
        for suffix in ("conv.weight", "conv.bias", "layer_norm.weight", "layer_norm.bias"):
            mapping[f"{fs2_prefix}.{suffix}"] = f"{hf_prefix}.{suffix}"
    for i in range(n_layers):
        fs2_prefix = f"encoder.layers.{i}"
        hf_prefix = f"wav2vec2.encoder.layers.{i}"
        for fs2_sub, hf_sub in _LAYER_SUBKEY_MAP.items():
            mapping[f"{fs2_prefix}.{fs2_sub}"] = f"{hf_prefix}.{hf_sub}"
    return mapping


def detect_arch(fs2_sd: Dict[str, Tensor]) -> Dict:
    layer_indices = {int(k.split(".")[2]) for k in fs2_sd if k.startswith("encoder.layers.")}
    n_layers = len(layer_indices)
    vocab_size = fs2_sd["final_proj.weight"].shape[0]
    hidden_size = fs2_sd["encoder.layer_norm.weight"].shape[0]
    intermediate_size = fs2_sd["encoder.layers.0.ffn.inner_proj.weight"].shape[0]
    num_attention_heads = hidden_size // 64  # head_dim=64 pour tous les OmniASR
    arch = dict(
        n_layers=n_layers, vocab_size=vocab_size, hidden_size=hidden_size,
        intermediate_size=intermediate_size, num_attention_heads=num_attention_heads,
    )
    logger.info("Architecture détectée : %s", arch)
    return arch


def build_hf_config(arch: Dict, ctc_blank_token_id: int):
    from transformers import Wav2Vec2Config
    return Wav2Vec2Config(
        hidden_size=arch["hidden_size"],
        num_hidden_layers=arch["n_layers"],
        num_attention_heads=arch["num_attention_heads"],
        intermediate_size=arch["intermediate_size"],
        hidden_act="gelu",
        do_stable_layer_norm=True,      # OmniASR = pre-norm, comme wav2vec2-large
        feat_extract_norm="layer",      # layer norm par couche conv (style v2)
        conv_dim=(512, 512, 512, 512, 512, 512, 512),
        conv_stride=(5, 2, 2, 2, 2, 2, 2),
        conv_kernel=(10, 3, 3, 3, 3, 2, 2),
        conv_bias=True,
        num_conv_pos_embeddings=128,
        num_conv_pos_embedding_groups=16,
        vocab_size=arch["vocab_size"],
        pad_token_id=ctc_blank_token_id,  # le pad_token double comme blank CTC dans HF
        ctc_loss_reduction="mean",
        add_adapter=False,
    )


def convert_state_dict(fs2_sd: Dict[str, Tensor], arch: Dict, ctc_blank_token_id: int):
    from transformers import Wav2Vec2ForCTC
    mapping = build_key_mapping(arch["n_layers"])
    config = build_hf_config(arch, ctc_blank_token_id)

    hf_sd: Dict[str, Tensor] = {}
    unmapped: List[str] = []
    for fs2_key, tensor in fs2_sd.items():
        if fs2_key in mapping:
            hf_sd[mapping[fs2_key]] = tensor.float().clone()
        else:
            unmapped.append(fs2_key)
    if unmapped:
        logger.warning("%d clés fairseq2 sans équivalent HF (ignorées) : %s", len(unmapped), unmapped)

    model = Wav2Vec2ForCTC(config)
    hf_expected = set(model.state_dict().keys())
    masked_key = "wav2vec2.masked_spec_embed"
    if masked_key in hf_expected and masked_key not in hf_sd:
        hf_sd[masked_key] = torch.zeros(config.hidden_size)

    missing, unexpected = model.load_state_dict(hf_sd, strict=False)
    if missing:
        logger.warning("Clés HF non chargées : %s", missing)
    if unexpected:
        logger.warning("Clés en trop (ignorées) : %s", unexpected)
    logger.info(
        "Chargé %d / %d paramètres HF depuis le checkpoint fairseq2.",
        len(hf_expected) - len(missing), len(hf_expected),
    )
    model.eval()
    return model


# ─────────────────────────────────────────────────────────────────────────
# 2. Reconstruction du tokenizer HF à partir du tokenizer fairseq2
# ─────────────────────────────────────────────────────────────────────────
def build_hf_tokenizer(fairseq2_tokenizer, output_dir: str) -> int:
    spm = fairseq2_tokenizer._model
    vi = fairseq2_tokenizer.vocab_info

    vocab: Dict[str, int] = {spm.index_to_token(i): i for i in range(spm.vocabulary_size)}
    blank_id = vi.bos_idx  # <s> = 0 = blank CTC (comme le blank par défaut de torch.ctc_loss)

    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    (out / "vocab.json").write_text(json.dumps(vocab, ensure_ascii=False, indent=2), encoding="utf-8")

    tokenizer_config = {
        "tokenizer_class": "Wav2Vec2CTCTokenizer",
        "unk_token": "<unk>",
        "bos_token": "<s>",
        "eos_token": "</s>",
        "pad_token": "<s>",
        "word_delimiter_token": "▁",
        "do_lower_case": False,
        "replace_word_delimiter_char": " ",
    }
    (out / "tokenizer_config.json").write_text(json.dumps(tokenizer_config, indent=2), encoding="utf-8")

    special_tokens_map = {"bos_token": "<s>", "eos_token": "</s>", "unk_token": "<unk>", "pad_token": "<s>"}
    (out / "special_tokens_map.json").write_text(json.dumps(special_tokens_map, indent=2), encoding="utf-8")

    preprocessor_config = {
        "feature_extractor_type": "Wav2Vec2FeatureExtractor",
        "feature_size": 1,
        "sampling_rate": 16000,
        "padding_value": 0.0,
        "do_normalize": True,
        "return_attention_mask": False,
    }
    (out / "preprocessor_config.json").write_text(json.dumps(preprocessor_config, indent=2), encoding="utf-8")

    logger.info("Fichiers tokenizer écrits dans %s (vocab=%d, blank=%d)", output_dir, spm.vocabulary_size, blank_id)
    return blank_id


# ─────────────────────────────────────────────────────────────────────────
# 3. Programme principal — réutilise les variables de votre notebook
# ─────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Variables supposées déjà définies plus haut dans le notebook :
    #   checkpoint_1000 : Path vers sdp_00.pt du checkpoint 1000 pas (cellule 30)
    #   tokenizer       : tokenizer fairseq2 omniASR_tokenizer_v1 (cellule 6)
    #   TRAIN_OUTPUT     : dossier de sortie de l'entraînement (réassigné à "-1000" en cellule 27)
    #   REPO_1000        : "Tree-AI-lab/omniASR-CTC-300M-baoule-1000steps" (cellule 35)
    #   finetuned_pipeline_1000 : pipeline fairseq2 chargé avec le checkpoint 1000 (cellule 30)
    #
    # -> Convertit la version 1000 pas (meilleur WER : 35.15 % contre 36.76 % pour la 500 pas).
    # Coller cette cellule APRÈS la cellule 30 (celle qui charge finetuned_pipeline_1000).

    checkpoint_path = checkpoint_1000

    HF_OUTPUT_DIR = TRAIN_OUTPUT.parent / "omniASR-CTC-300M-baoule-1000-hf"
    PUSH_TO_HUB = True          # mettre False pour ne convertir qu'en local
    HF_REPO_ID = REPO_1000      # même dépôt que la version fairseq2 1000 pas, ou un nouveau nom

    logger.info("Chargement du checkpoint fairseq2 : %s", checkpoint_path)
    fs2_state_dict = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

    arch = detect_arch(fs2_state_dict)
    blank_id = build_hf_tokenizer(tokenizer, str(HF_OUTPUT_DIR))
    hf_model = convert_state_dict(fs2_state_dict, arch, ctc_blank_token_id=blank_id)

    hf_model.save_pretrained(HF_OUTPUT_DIR, safe_serialization=True)
    logger.info("Modèle transformers sauvegardé dans %s", HF_OUTPUT_DIR)

    # ── Test de parité rapide : compare fairseq2 vs HF sur le même audio ──
    from transformers import Wav2Vec2ForCTC, Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

    hf_model_reloaded = Wav2Vec2ForCTC.from_pretrained(HF_OUTPUT_DIR)
    hf_tok = Wav2Vec2CTCTokenizer.from_pretrained(HF_OUTPUT_DIR)
    hf_fe = Wav2Vec2FeatureExtractor.from_pretrained(HF_OUTPUT_DIR)
    processor = Wav2Vec2Processor(feature_extractor=hf_fe, tokenizer=hf_tok)
    processor.save_pretrained(HF_OUTPUT_DIR)

    import soundfile as sf
    test_parquet_row = None
    test_dir = DATASET_VERSION / "corpus=waxal" / "split=test" / "language=bci_Latn"
    import pyarrow.parquet as pq
    for parquet_file in sorted(test_dir.glob("*.parquet")):
        table = pq.read_table(parquet_file, columns=["text", "audio_bytes", "audio_size"])
        for row in table.to_pylist():
            if row["audio_size"] <= 40 * 16_000:
                test_parquet_row = row
                break
        if test_parquet_row:
            break

    if test_parquet_row is not None:
        import numpy as np
        audio_bytes = bytes(v % 256 for v in test_parquet_row["audio_bytes"])
        import io
        waveform, sr = sf.read(io.BytesIO(audio_bytes), dtype="float32")
        inputs = processor(waveform, sampling_rate=16000, return_tensors="pt")
        with torch.no_grad():
            logits = hf_model_reloaded(**inputs).logits
        pred_ids = torch.argmax(logits, dim=-1)
        hf_transcript = processor.batch_decode(pred_ids)[0]
        print("Référence     :", test_parquet_row["text"])
        print("HF transformers:", hf_transcript)
        print("fairseq2 (pipeline déjà chargé) :",
              finetuned_pipeline_1000.transcribe([audio_bytes], batch_size=1)[0])
    else:
        logger.warning("Aucun audio de test trouvé pour la vérification de parité.")

    # ── Publication sur le Hub (mêmes fichiers que la version fairseq2 + les nouveaux) ──
    if PUSH_TO_HUB:
        from huggingface_hub import HfApi
        api = HfApi()
        api.create_repo(HF_REPO_ID, repo_type="model", private=True, exist_ok=True)
        hf_model_reloaded.push_to_hub(HF_REPO_ID, commit_message="Ajout du modèle natif transformers (Wav2Vec2ForCTC)")
        processor.push_to_hub(HF_REPO_ID, commit_message="Ajout du tokenizer/feature extractor transformers")
        logger.info("Modèle transformers publié sur https://huggingface.co/%s", HF_REPO_ID)
        print(
            "\nUtilisation en 3 lignes désormais possible :\n\n"
            "from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor\n"
            f'processor = Wav2Vec2Processor.from_pretrained("{HF_REPO_ID}")\n'
            f'model     = Wav2Vec2ForCTC.from_pretrained("{HF_REPO_ID}")\n'
        )